# Step 9: Deploy the Model

**SageMaker Unified Studio Component**: Inference Endpoints

In [ ]:
import sagemaker
import os
from sagemaker.sklearn import SKLearnModel
from dotenv import load_dotenv

load_dotenv()
bucket_name = os.getenv('BUCKET_NAME')

# Use SageMaker's execution role (recommended for Unified Studio)
try:
    role = sagemaker.get_execution_role()
    print(f"Using SageMaker execution role: {role}")
except ValueError:
    # Fallback to .env file role if running locally
    role = os.getenv('EXECUTION_ROLE')
    print(f"Using .env execution role: {role}")

## Deploy Endpoint

In [ ]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
from botocore.exceptions import ClientError

model_data = f's3://{bucket_name}/models/logistic_regression/model.tar.gz'
endpoint_name = 'machine-overheat-endpoint'

try:
    sklearn_model = SKLearnModel(
        model_data=model_data,
        role=role,
        entry_point='inference.py',
        framework_version='1.2-1',
        py_version='py3'
    )

    predictor = sklearn_model.deploy(
        initial_instance_count=1,
        instance_type='ml.t2.medium',
        endpoint_name=endpoint_name
    )
    print(f"✓ Endpoint deployed: {predictor.endpoint_name}")

except ClientError as e:
    if 'already existing' in str(e).lower() or 'Cannot create already existing' in str(e):
        print(f"Endpoint '{endpoint_name}' already exists. Connecting to it...")
        predictor = Predictor(
            endpoint_name=endpoint_name,
            serializer=JSONSerializer(),
            deserializer=JSONDeserializer()
        )
        print(f"✓ Connected to existing endpoint: {endpoint_name}")
    else:
        raise e

## Test the Endpoint

In [ ]:
# Test with a normal temperature (should NOT overheat)
test_input = {'temperature': 78, 'room_temp': 25}
response = predictor.predict(test_input)
print(f"Input: {test_input}")
print(f"Prediction: {response}")
print(f"→ Temperature {test_input['temperature']}°C: {'OVERHEAT!' if response['prediction'] == 1 else 'Normal'}")

In [ ]:
# Test with a high temperature (SHOULD overheat)
test_input = {'temperature': 85, 'room_temp': 25}
response = predictor.predict(test_input)
print(f"Input: {test_input}")
print(f"Prediction: {response}")
print(f"→ Temperature {test_input['temperature']}°C: {'OVERHEAT!' if response['prediction'] == 1 else 'Normal'}")

## Cleanup (Optional)

In [ ]:
# predictor.delete_endpoint()
# print("Endpoint deleted")